# 1. IAM

Identity and Access Management - AWS IAM (Identity and Access Management) is used to control who can access your AWS resources and what they are allowed to do

- Global not regional tool
- We can create users and groups
- Groups will have same type of permissions
- All the permissions are created in Json file
- We can create custom json files for the permissions (policy)

And then we use users cred to do the work.

Another thing is to create Role:

Suppose we want to get access of S3 in EC2, do we have to save access_key there? not necessary

We can give EC2 a role of IAM and then based on the policies it will work without creds.


# 2. EC2 (Elastic Compute Cloud)

Virtual machines in the cloud.

What it is: EC2 lets you rent virtual servers ('instances') with configurable CPU, RAM, storage, and OS. You choose an instance type and pay for what you use.

Why it matters: It's the most fundamental compute building block on AWS — the direct cloud equivalent of a physical or virtual server you'd otherwise manage yourself.

Key concepts / terms:
-	AMI (Amazon Machine Image) — the template (OS + software) an instance is launched from
-	Instance type — hardware profile, e.g. t3.micro (small/cheap) vs m5.large (general purpose)
-	Security Group — a virtual firewall attached to the instance
-	Key pair — SSH credentials used to log into the instance
-	Elastic IP — a static public IP address you can attach to an instance

Example use case: Hosting a backend API or a database server that needs full control over the OS.




# S3 (Simple Storage Service)

Object storage — files, not a filesystem.

What it is: S3 stores files ('objects') inside 'buckets'. It's not a traditional disk — you don't attach it to a server; you access objects over HTTP via an API/SDK.

Why it matters: It's cheap, virtually unlimited, and highly durable (11 nines of durability). It's used everywhere — backups, static websites, data lakes, ML training data, logs.

Key concepts / terms:
-	Bucket — a top-level container for objects, globally unique name
-	Object — a file plus metadata, identified by a key (path-like string)
-	Storage class — Standard, Infrequent Access, Glacier (cold storage), etc. — trade cost vs retrieval speed
-	Bucket policy / ACL — controls who can read/write objects
-	Versioning & lifecycle rules — keep object history, auto-archive or delete old data

Example use case: Storing training datasets and model artifacts for a machine learning pipeline, or hosting a static React website.


# 4. ECR (Elastic Container Registry)

A private Docker Hub for your AWS account.

What it is: ECR stores and versions Docker container images so they can be pulled by ECS, EKS, or Lambda.

Why it matters: Any containerized workload on AWS needs somewhere to store its images — ECR is the native, IAM-integrated option (vs. using public Docker Hub).

Key concepts / terms:
-	Repository — like a Docker Hub repo, holds tagged versions of one image
-	Image tag — a version label (e.g. v1.2, latest)
-	Push/pull — done via Docker CLI after authenticating with IAM
-	Image scanning — automatic vulnerability scanning of stored images

Example use case: A CI/CD pipeline builds a Docker image, pushes it to ECR, then ECS pulls that exact image to deploy.


# 5. RDS (Relational Database Service)

Managed SQL databases — AWS handles the boring parts.

What it is: RDS runs relational databases (PostgreSQL, MySQL, MariaDB, SQL Server, Oracle) for you — AWS handles patching, backups, and failover.

Why it matters: Running a production database yourself means managing backups, replication, patching, and scaling. RDS automates all of that so you focus on your data model, not server admin.

Key concepts / terms:
-	Instance class — the compute/memory size of the database server
-	Multi-AZ — a standby replica in another Availability Zone for automatic failover
-	Read replica — a read-only copy to scale read traffic
-	Automated backups & snapshots — point-in-time recovery
-	Parameter group — database engine configuration settings

Example use case: An e-commerce app storing orders, users, and inventory in a PostgreSQL database managed by RDS.


In [ ]:
import psycopg2
import boto3

auth_token = boto3.client('rds', region_name='us-east-1').generate_db_auth_token(DBHostname='database1.cw5amkiows2d.us-east-1.rds.amazonaws.com', Port=5432, DBUsername='postgres', Region='us-east-1')

conn = None
try:
    conn = psycopg2.connect(
        host='database1.cw5amkiows2d.us-east-1.rds.amazonaws.com',
        port=5432,
        database='postgres',
        user='postgres',
        password=auth_token,
        sslmode='verify-full',
    sslrootcert='./global-bundle.pem'
    )
    cur = conn.cursor()
    cur.execute('SELECT version();')
    print(cur.fetchone()[0])
    cur.close()
except Exception as e:
    print(f"Database error: {e}")
    raise
finally:
    if conn:
        conn.close()

Database error: connection to server at "database1.cw5amkiows2d.us-east-1.rds.amazonaws.com" (52.71.139.163), port 5432 failed: FATAL:  password authentication failed for user "postgres"
connection to server at "database1.cw5amkiows2d.us-east-1.rds.amazonaws.com" (52.71.139.163), port 5432 failed: FATAL:  no pg_hba.conf entry for host "45.119.31.38", user "postgres", database "postgres", no encryption



OperationalError: connection to server at "database1.cw5amkiows2d.us-east-1.rds.amazonaws.com" (52.71.139.163), port 5432 failed: FATAL:  password authentication failed for user "postgres"
connection to server at "database1.cw5amkiows2d.us-east-1.rds.amazonaws.com" (52.71.139.163), port 5432 failed: FATAL:  no pg_hba.conf entry for host "45.119.31.38", user "postgres", database "postgres", no encryption


# 6. VPC (Virtual Private Cloud)
Your own private network inside AWS.

What it is: A VPC is an isolated virtual network where you place your AWS resources. You control the IP range, subnets, routing, and what can talk to what.

Why it matters: Understanding VPC is what separates 'I can click buttons in AWS' from 'I understand how AWS actually works.' Almost every real architecture question comes back to networking.

Key concepts / terms:
-	CIDR block — the IP address range for the VPC (e.g. 10.0.0.0/16)
-	Subnet — a smaller network segment within a VPC, tied to one Availability Zone
-	Route table — rules that decide where network traffic is directed
-	Internet Gateway — allows a VPC to reach the public internet
-	NAT Gateway — lets private subnets reach the internet outbound without being publicly reachable

Example use case: A production app where web servers sit in a public subnet, but the database sits in a private subnet with no direct internet access.


# 7. CloudWatch

Eyes and ears for everything running on AWS.

What it is: CloudWatch collects logs, metrics, and events from AWS services, and lets you set alarms and dashboards on them.

Why it matters: You can't fix or scale what you can't see. CloudWatch is how you know a server is overloaded, an app is throwing errors, or a Lambda function is failing.

Key concepts / terms:
-   Metric — a numeric data point over time (e.g. CPU utilization)
-	Log group / log stream — where application and service logs are collected
-	Alarm — triggers an action (e.g. notify, auto-scale) when a metric crosses a threshold
-	Dashboard — a visual summary of key metrics

Example use case: An alarm triggers an auto-scaling event when average CPU across EC2 instances exceeds 70% for 5 minutes.


# 8. Lambda

Run code without managing any servers.

What it is: Lambda runs your code in response to an event (an HTTP request, a file upload, a schedule) and you're billed only for the compute time 
actually used — no server to provision or patch.

Why it matters: It's the core of 'serverless' architecture — ideal for small, event-driven tasks and APIs with unpredictable or spiky traffic.

Key concepts / terms:
-	Trigger — the event source that invokes the function (API Gateway, S3, EventBridge, etc.)
-	Cold start — the delay when a function runs after being idle
-	Execution role — the IAM role the function assumes to access other AWS resources
-	Timeout & memory — you configure both; billing is based on execution time × memory

Example use case: Resizing an image automatically the moment it's uploaded to an S3 bucket, with no server running 24/7.
